# Módulo 04 · Dilema del Prisionero
**Teoría de Juegos — Tutorial Interactivo**

El dilema del prisionero es el ejemplo más citado de la teoría de juegos. Analizamos el juego puntual y el iterado, implementamos estrategias y corremos el torneo de Axelrod.

In [ ]:
import numpy as np
import nashpy as nash
import axelrod
import matplotlib.pyplot as plt

print(f'axelrod {axelrod.__version__}')

## 1. El juego puntual: análisis con nashpy

In [ ]:
# Pagos: años evitados de condena (más es mejor)
# Acciones: 0=Cooperar (callar), 1=Traicionar (confesar)
A = np.array([[3, 0],
              [5, 1]])
B = np.array([[3, 5],
              [0, 1]])

pd = nash.Game(A, B)
eqs = list(pd.support_enumeration())
print('Equilibrios de Nash:')
for s1, s2 in eqs:
    labels = ['Coop', 'Trai']
    a1 = labels[np.argmax(s1)]
    a2 = labels[np.argmax(s2)]
    print(f'  ({a1}, {a2}) — pagos {pd[s1, s2]}')

print('\n¿Por qué Traicionar es dominante?')
print('Si J2 coopera:    Traicionar da', A[1,0], 'vs Cooperar', A[0,0])
print('Si J2 traiciona:  Traicionar da', A[1,1], 'vs Cooperar', A[0,1])
print('→ Traicionar es siempre mejor. Es estrategia dominante.')

## 2. Implementar Tit-for-Tat desde cero

In [ ]:
PAYOFFS = {'CC': (3,3), 'CD': (0,5), 'DC': (5,0), 'DD': (1,1)}

def tit_for_tat(history):
    """Coopera primero. Luego imita la última jugada del rival."""
    if not history:
        return 'C'
    return history[-1][1]  # última jugada del rival

def always_defect(history):
    return 'D'

def grim_trigger(history):
    """Coopera hasta la primera traición. Luego siempre D."""
    if any(h[1] == 'D' for h in history):
        return 'D'
    return 'C'

def pavlov(history):
    """Win-Stay, Lose-Shift."""
    if not history:
        return 'C'
    last_my, last_opp = history[-1]
    pay = PAYOFFS[last_my + last_opp][0]
    return last_my if pay >= 3 else ('D' if last_my == 'C' else 'C')

def play_match(strat_a, strat_b, rounds=20):
    """Juega un partido entre dos estrategias."""
    hist_a, hist_b = [], []  # perspectiva de cada jugador
    score_a, score_b = 0, 0
    for _ in range(rounds):
        a = strat_a(hist_a)
        b = strat_b(hist_b)
        pa, pb = PAYOFFS[a + b]
        score_a += pa; score_b += pb
        hist_a.append((a, b))
        hist_b.append((b, a))
    return score_a, score_b

# Test
strategies = {'Tit-for-Tat': tit_for_tat, 'Always Defect': always_defect,
              'Grim Trigger': grim_trigger, 'Pavlov': pavlov}

s1, s2 = play_match(tit_for_tat, always_defect, 20)
print(f'Tit-for-Tat vs Always Defect (20 rondas): {s1} vs {s2}')

## 3. Torneo round-robin

In [ ]:
def run_tournament(strategies, rounds=50):
    names = list(strategies.keys())
    totals = {n: 0 for n in names}
    
    for i, n1 in enumerate(names):
        for j, n2 in enumerate(names):
            if i == j:
                continue
            s1, s2 = play_match(strategies[n1], strategies[n2], rounds)
            totals[n1] += s1
            totals[n2] += s2
    
    ranking = sorted(totals.items(), key=lambda x: -x[1])
    return ranking

ranking = run_tournament(strategies, rounds=50)
print('=== Torneo round-robin (50 rondas por emparejamiento) ===')
for pos, (name, score) in enumerate(ranking, 1):
    print(f'{pos}. {name:20s} — {score} puntos')

## 4. Torneo con la librería axelrod (reproducción del experimento original)

In [ ]:
players = [
    axelrod.TitForTat(),
    axelrod.Defector(),
    axelrod.Cooperator(),
    axelrod.TitFor2Tats(),
    axelrod.GrimTrigger(),
    axelrod.WinStayLoseShift(),
    axelrod.Random(),
]

tournament = axelrod.Tournament(
    players, turns=200, repetitions=5, seed=42, with_morality=False
)
results = tournament.play(progress_bar=False)

print('Ranking del torneo Axelrod:')
for i, name in enumerate(results.ranked_names, 1):
    print(f'  {i}. {name}')

In [ ]:
# Visualizar scores medios
fig, ax = plt.subplots(figsize=(8, 4))
scores = [results.normalised_scores[results.players.index(
    next(p for p in players if str(p) == name))] 
          for name in results.ranked_names]
means = [np.mean(s) for s in scores]
colors = ['#2d7a50' if i == 0 else '#1a3a5c' for i in range(len(means))]
ax.barh(results.ranked_names[::-1], means[::-1], color=colors[::-1])
ax.set_xlabel('Pago medio normalizado por ronda')
ax.set_title('Torneo de Axelrod — puntuación media')
plt.tight_layout()
plt.show()

## 5. Ejercicios

In [ ]:
# EJERCICIO 1 — Implementa tu propia estrategia y añádela al torneo
def mi_estrategia(history):
    """
    Implementa aquí tu estrategia.
    history: lista de tuplas (mi_jugada, jugada_rival)
    Devuelve 'C' o 'D'
    """
    # TU CÓDIGO AQUÍ:
    pass

# Cuando esté lista, añádela:
# strategies['Mi Estrategia'] = mi_estrategia
# ranking_nuevo = run_tournament(strategies, rounds=50)
# print(ranking_nuevo)

In [ ]:
# EJERCICIO 2 — ¿Qué pasa cuando el horizonte es conocido?
# Juega TfT vs Always Defect con 20 rondas conocidas y 20 desconocidas.
# ¿Cambia el resultado? ¿Por qué?
# TU CÓDIGO AQUÍ:
